In [86]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("ihasan88/student-performance-dataset")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\Antek\.cache\kagglehub\datasets\ihasan88\student-performance-dataset\versions\3


In [87]:
path_to_csv = os.path.join(path, "bangladesh_student_performance.csv")
data = pd.read_csv(path_to_csv, true_values=['Yes'], false_values=['No'])

In [88]:
data.head()

,Student_ID,Gender,Age,District,School_Type,Study_Hours_per_Week,Attendance,Parent_Education,Family_Income_BDT,Internet_Access,Private_Tuition,Previous_GPA,SSC_Result,HSC_Result
0,1,Male,16,Rangpur,Private,11,70,Graduate,49169,True,True,4.83,4.22,5.00
1,2,Female,17,Rajshahi,Private,7,76,Primary,57947,False,False,4.96,4.09,4.77
2,3,Male,15,Barisal,Private,27,63,Higher Secondary,57224,True,True,4.32,4.86,4.62
3,4,Male,15,Mymensingh,Private,18,84,Primary,55865,False,False,4.63,3.54,3.75
4,5,Male,15,Mymensingh,Public,28,89,Higher Secondary,35553,False,True,4.50,4.65,3.46


In [89]:
X = data.drop(['Student_ID', 'HSC_Result'], axis=1)  # predictors
y = data.HSC_Result  # target

In [90]:
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(exclude=['int64', 'float64']).columns

In [91]:
cat_nominal_cols = ['Gender', 'District', 'School_Type', 'Internet_Access', 'Private_Tuition']
cat_ordinal_cols = ['Parent_Education']

education_ladder = ['Primary', 'Secondary', 'Higher Secondary', 'Graduate']

In [92]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

numerical_transformer = SimpleImputer(strategy='median')

cat_nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

cat_ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal_encoder', OrdinalEncoder(categories=[education_ladder], handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat_nominal', cat_nominal_transformer, cat_nominal_cols),
    ('cat_ordinal', cat_ordinal_transformer, cat_ordinal_cols),
])

In [93]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=0)

my_pipepline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model),
])

In [94]:
from sklearn.model_selection import cross_val_score

scores = -1 * cross_val_score(my_pipepline, X, y, scoring='neg_mean_absolute_error', cv=5)
print('Score:', scores.mean())

Score: 0.5244909
